# Model B: Time-Series + NLP Dasar (VADER + Loughran-McDonald)

Model B = Model A + fitur sentimen lexicon Tugas 2. Teks berita tidak diproses ulang
di sini; yang dipakai adalah agregat harian yang sudah disejajarkan ke tanggal bursa
(WIB) oleh pipeline tim.

| Varian | Fitur teks | n fitur |
|---|---|---|
| `B_vader` | VADER judul: mean, min, max, std, rasio positif, rasio negatif | 7 + 6 = 13 |
| `B_lm` | LM judul dan LM isi, 5 kolom masing-masing | 7 + 10 = 17 |
| `B_lex` | VADER + LM judul + LM isi | 7 + 16 = 23 |

Split diambil apa adanya dari kolom `split` di `prices_daily.parquet`, sehingga
baris yang dipakai sama dengan Model A dan C.

## 0. Import dan lokasi file

In [1]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import sklearn, xgboost, lightgbm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    AdaBoostRegressor, GradientBoostingRegressor,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED = 42
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.6g}")


def find_root(start):
    # Notebook bisa dijalankan dari folder mana pun, jadi root dicari ke atas.
    for cand in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (cand / "data" / "export" / "prices_daily.parquet").exists():
            return cand
    raise FileNotFoundError("Folder proyek (berisi data/export/prices_daily.parquet) tidak ditemukan")


ROOT = find_root(Path.cwd())
EXPORT_DIR = ROOT / "data" / "export"
OUT_DIR = ROOT / "trial-and-error" / "Ken" / "results"
sys.path.insert(0, str((ROOT / "src").resolve()))
from eval_utils import evaluate, METRICS

print("Proyek :", ROOT)
print("Metrik :", METRICS, "| sklearn", sklearn.__version__)

Proyek : D:\nlpproject-grestercinta
Metrik : ['MAE', 'MSE', 'RMSE', 'DirAcc'] | sklearn 1.7.2


## 1. Kurs harian dan sentimen harian

In [2]:
prices = pd.read_parquet(EXPORT_DIR / "prices_daily.parquet")
sent = pd.read_parquet(EXPORT_DIR / "sentiment_daily.parquet")

prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()
sent["date"] = pd.to_datetime(sent["date"]).dt.normalize()

missing = ~prices["date"].isin(sent["date"])
print("baris kurs      :", len(prices))
print("baris sentimen  :", len(sent))
print("tanggal kurs tanpa sentimen:", int(missing.sum()))

df = prices.merge(sent, on="date", how="left", validate="one_to_one")
print("setelah merge   :", len(df), "| kolom:", len(df.columns))
print("rentang         :", df["date"].min().date(), "->", df["date"].max().date())
print("NaN per kolom   :", int(df.isna().sum().sum()))

baris kurs      : 1194
baris sentimen  : 1202
tanggal kurs tanpa sentimen: 0
setelah merge   : 1194 | kolom: 31
rentang         : 2021-09-08 -> 2026-08-31
NaN per kolom   : 0


## 2. Kelompok fitur

In [3]:
FEAT_A = [c for c in prices.columns if c.startswith("kurs_lag") or c.startswith("kurs_roll_")]
FEAT_VADER = [c for c in sent.columns if c.startswith("vader_")]
FEAT_LM = [c for c in sent.columns if c.startswith(("lm_title_", "lm_text_"))]

VARIANTS = {
    "A": FEAT_A,
    "B_vader": FEAT_A + FEAT_VADER,
    "B_lm": FEAT_A + FEAT_LM,
    "B_lex": FEAT_A + FEAT_VADER + FEAT_LM,
}

print("fitur A    :", len(FEAT_A), FEAT_A)
print("fitur VADER:", len(FEAT_VADER))
print("fitur LM   :", len(FEAT_LM))
for name, cols in VARIANTS.items():
    print(f"{name:8}: {len(cols):2d} fitur")

fitur A    : 7 ['kurs_lag1', 'kurs_lag2', 'kurs_lag3', 'kurs_lag4', 'kurs_lag5', 'kurs_roll_mean5', 'kurs_roll_std5']
fitur VADER: 6
fitur LM   : 10
A       :  7 fitur
B_vader : 13 fitur
B_lm    : 17 fitur
B_lex   : 23 fitur


## 3. Split dari kolom `split`

In [4]:
train = df[df["split"] == "train"].reset_index(drop=True)
val = df[df["split"] == "val"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)

for name, part in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"{name:5}: {len(part):4d} | {part['date'].min().date()} -> {part['date'].max().date()}")

ytr, yva, yte = train["target"].values, val["target"].values, test["target"].values
print("total baris:", len(train) + len(val) + len(test))

Train:  836 | 2021-09-08 -> 2025-02-18
Val  :  178 | 2025-02-20 -> 2025-11-20
Test :  180 | 2025-11-24 -> 2026-08-31
total baris: 1194


## 4. Cek kesesuaian dengan `dataset_model_A.parquet`

In [5]:
a = pd.read_parquet(OUT_DIR / "dataset_model_A.parquet")
a["date"] = pd.to_datetime(a["Tanggal_dt"]).dt.normalize()
cea = df[["date", "target"]].merge(a[["date", "target"]], on="date", suffixes=("", "_A"))

print("baris B         :", len(df))
print("baris A         :", len(a))
print("tanggal cocok   :", len(cea))
print("selisih target maks:", float((cea["target"] - cea["target_A"]).abs().max()))
print("split A         :", a["split"].value_counts().to_dict())
print("split B         :", df["split"].value_counts().to_dict())

baris B         : 1194
baris A         : 1194
tanggal cocok   : 1194
selisih target maks: 0.0
split A         : {'train': 836, 'test': 180, 'val': 178}
split B         : {'train': 836, 'test': 180, 'val': 178}


## 5. Model dan evaluasi

In [6]:
MODELS = {
    "LinearRegression": lambda: LinearRegression(),
    "Ridge":            lambda: Ridge(),
    "Lasso":            lambda: Lasso(),
    "ElasticNet":       lambda: ElasticNet(),
    "BayesianRidge":    lambda: BayesianRidge(),
    "KNN":              lambda: KNeighborsRegressor(),
    "SVR":              lambda: SVR(),
    "DecisionTree":     lambda: DecisionTreeRegressor(random_state=SEED),
    "RandomForest":     lambda: RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "ExtraTrees":       lambda: ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    "AdaBoost":         lambda: AdaBoostRegressor(random_state=SEED),
    "GradientBoosting": lambda: GradientBoostingRegressor(random_state=SEED),
    "XGBoost":          lambda: XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0),
    "LightGBM":         lambda: LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1),
}


def scale_features(cols):
    # Scaler hanya melihat train, val dan test ikut transformasi yang sama.
    scaler = StandardScaler().fit(train[cols])
    return (scaler.transform(train[cols]),
            scaler.transform(val[cols]),
            scaler.transform(test[cols]))


def collect_prediction(preds, varian, model_name, split, part, y_true, y_pred):
    preds.append(pd.DataFrame({
        "Varian": varian, "Model": model_name, "Split": split,
        "date": part["date"].values, "y_true": y_true, "y_pred": y_pred,
    }))


rows, preds = [], []
for varian, cols in VARIANTS.items():
    Xtr, Xva, Xte = scale_features(cols)
    for model_name, make_model in MODELS.items():
        model = make_model().fit(Xtr, ytr)
        for split, X, y, part in [("Val", Xva, yva, val), ("Test", Xte, yte, test)]:
            yp = model.predict(X)
            rows.append({"Varian": varian, "Model": model_name, "Split": split,
                         "n_fitur": len(cols), **evaluate(y, yp)})
            collect_prediction(preds, varian, model_name, split, part, y, yp)

results = pd.DataFrame(rows)
results["Degenerate"] = results["PredStd"] < 1e-9
print("total baris hasil:", len(results))
print("model degenerate :", int(results["Degenerate"].sum()))

total baris hasil: 112
model degenerate : 24


## 6. Hasil per varian

In [7]:
for split in ("Val", "Test"):
    t = (results[(results["Split"] == split) & (~results["Degenerate"])]
         .sort_values(["Varian", "RMSE"])
         .groupby("Varian").head(3)
         .set_index(["Varian", "Model"])[["n_fitur", "RMSE", "MAE", "DirAcc", "PredStd"]])
    print(f"\n=== {split} | 3 model terbaik per varian ===")
    print(t.to_string())


=== Val | 3 model terbaik per varian ===
                          n_fitur       RMSE        MAE   DirAcc     PredStd
Varian  Model                                                               
A       BayesianRidge           7 0.00338869 0.00250102 0.448864  0.00025135
        Ridge                   7 0.00339263 0.00251092 0.460227 0.000353976
        LinearRegression        7 0.00340153 0.00252672 0.465909 0.000473432
B_lex   BayesianRidge          23  0.0034155 0.00250695 0.454545 0.000389637
        Ridge                  23 0.00343532 0.00254408 0.426136 0.000618727
        LinearRegression       23 0.00344437 0.00255784 0.426136 0.000700423
B_lm    BayesianRidge          17 0.00339795 0.00248806 0.488636 0.000316738
        Ridge                  17 0.00339926 0.00251338 0.482955 0.000471428
        LinearRegression       17 0.00340544  0.0025275 0.494318 0.000563956
B_vader RandomForest           13 0.00341481 0.00253223 0.505682 0.000674454
        BayesianRidge          13 

## 7. Delta RMSE terhadap Model A

In [8]:
base = (results[(results["Varian"] == "A") & (results["Split"] == "Test")]
        .set_index("Model")["RMSE"])

delta_rows = []
for var in ("B_vader", "B_lm", "B_lex"):
    sub = results[(results["Varian"] == var) & (results["Split"] == "Test") & (~results["Degenerate"])]
    best = sub.loc[sub["RMSE"].idxmin()]
    delta_rows.append({
        "Varian": var, "Model": best["Model"], "n_fitur": int(best["n_fitur"]),
        "RMSE": best["RMSE"], "DirAcc": best["DirAcc"],
        "dRMSE_pct": 100 * (best["RMSE"] - base[best["Model"]]) / base[best["Model"]],
    })

print("=== Model terbaik per varian B (Test) ===")
print(pd.DataFrame(delta_rows).to_string(index=False, float_format=lambda x: f"{x:.6g}"))

=== Model terbaik per varian B (Test) ===
 Varian            Model  n_fitur       RMSE   DirAcc  dRMSE_pct
B_vader LinearRegression       13  0.0029207      0.5    -2.4668
   B_lm    BayesianRidge       17 0.00296399 0.443182  -0.673353
  B_lex LinearRegression       23 0.00292131 0.556818   -2.44642


## 8. Hyperparameter tuning

In [9]:
GRIDS = {
    "Ridge": {"alpha": [0.1, 1.0, 10.0, 100.0]},
    "Lasso": {"alpha": [1e-5, 1e-4, 1e-3, 1e-2]},
    "ElasticNet": {"alpha": [1e-5, 1e-4, 1e-3], "l1_ratio": [0.2, 0.5, 0.8]},
    "KNN": {"n_neighbors": [3, 5, 7, 11, 21, 41], "weights": ["uniform", "distance"]},
    "SVR": {"C": [0.1, 1.0, 10.0], "epsilon": [1e-4, 1e-3], "kernel": ["rbf"]},
    "RandomForest": {"max_depth": [3, 5, None], "min_samples_leaf": [1, 5]},
    "XGBoost": {"max_depth": [2, 3], "learning_rate": [0.03, 0.05], "n_estimators": [300]},
    "LightGBM": {"num_leaves": [7, 15], "learning_rate": [0.03, 0.05], "n_estimators": [300]},
}
CV = TimeSeriesSplit(n_splits=5)

tuned = {}
tune_rows = []
for var in ("B_vader", "B_lm", "B_lex"):
    sub = results[(results["Varian"] == var) & (results["Split"] == "Val") & (~results["Degenerate"])]
    model_name = sub.sort_values("RMSE").iloc[0]["Model"]
    if model_name not in GRIDS:
        model_name = "Ridge"

    Xtr, Xva, Xte = scale_features(VARIANTS[var])
    gs = GridSearchCV(MODELS[model_name](), GRIDS[model_name], cv=CV, n_jobs=-1,
                      scoring="neg_root_mean_squared_error")
    gs.fit(Xtr, ytr)
    best = gs.best_estimator_
    tuned[var] = (best, VARIANTS[var])

    for split, X, y, part in [("Val", Xva, yva, val), ("Test", Xte, yte, test)]:
        yp = best.predict(X)
        tune_rows.append({"Varian": var + "_tuned", "Model": model_name + "+tuned",
                          "Split": split, "n_fitur": len(VARIANTS[var]),
                          "params": json.dumps(gs.best_params_),
                          "cv_RMSE": float(-gs.best_score_), **evaluate(y, yp)})
        collect_prediction(preds, var + "_tuned", model_name + "+tuned", split, part, y, yp)

tuning = pd.DataFrame(tune_rows)
tuning["Degenerate"] = tuning["PredStd"] < 1e-9
print(tuning[tuning["Split"] == "Test"][["Varian", "Model", "params", "cv_RMSE", "RMSE", "DirAcc"]]
      .to_string(index=False, float_format=lambda x: f"{x:.6g}"))

banding = []
for var in ("B_vader", "B_lm", "B_lex"):
    d = results[(results["Varian"] == var) & (results["Split"] == "Test")
                & (~results["Degenerate"])].sort_values("RMSE").iloc[0]
    t = tuning[(tuning["Varian"] == var + "_tuned") & (tuning["Split"] == "Test")].iloc[0]
    banding.append({"Varian": var, "RMSE_default": d["RMSE"], "RMSE_tuned": t["RMSE"],
                    "perubahan_pct": 100 * (t["RMSE"] - d["RMSE"]) / d["RMSE"]})
print("\n=== Default vs tuned (Test) ===")
print(pd.DataFrame(banding).to_string(index=False, float_format=lambda x: f"{x:.6g}"))

       Varian              Model                                  params    cv_RMSE       RMSE   DirAcc
B_vader_tuned RandomForest+tuned {"max_depth": 3, "min_samples_leaf": 5} 0.00393498 0.00297987 0.431818
   B_lm_tuned        Ridge+tuned                        {"alpha": 100.0} 0.00366886  0.0029626 0.448864
  B_lex_tuned        Ridge+tuned                        {"alpha": 100.0} 0.00367316 0.00294056 0.454545

=== Default vs tuned (Test) ===
 Varian  RMSE_default  RMSE_tuned  perubahan_pct
B_vader     0.0029207  0.00297987        2.02573
   B_lm    0.00296399   0.0029626      -0.047024
  B_lex    0.00292131  0.00294056       0.658961


## 9. Permutation importance

In [10]:
imp_rows = []
for var, (model, cols) in tuned.items():
    Xva, _ = scale_features(cols)[1:]
    r = permutation_importance(model, Xva, yva, n_repeats=20, random_state=SEED,
                               scoring="neg_root_mean_squared_error")
    for feat, mean, std in sorted(zip(cols, r.importances_mean, r.importances_std),
                                  key=lambda t: -t[1]):
        imp_rows.append({"Varian": var, "Fitur": feat, "importance": mean, "std": std})

imp = pd.DataFrame(imp_rows)
for var in tuned:
    print(f"\n=== {var}: 8 fitur paling berpengaruh (validasi) ===")
    print(imp[imp["Varian"] == var].head(8).to_string(index=False, float_format=lambda x: f"{x:.6g}"))


=== B_vader: 8 fitur paling berpengaruh (validasi) ===
 Varian                 Fitur   importance         std
B_vader      vader_title_mean  2.05036e-05 1.90431e-05
B_vader       vader_title_min  3.79171e-06 6.91564e-06
B_vader             kurs_lag1  3.51444e-06 4.50335e-06
B_vader vader_title_pos_ratio  2.32351e-06 1.76688e-06
B_vader             kurs_lag3  6.23219e-07 1.07492e-06
B_vader       kurs_roll_mean5  1.98904e-07 1.22593e-06
B_vader             kurs_lag4  1.98429e-07  2.1304e-06
B_vader vader_title_neg_ratio -2.28879e-07 3.18679e-06

=== B_lm: 8 fitur paling berpengaruh (validasi) ===
Varian             Fitur  importance         std
  B_lm lm_text_litigious 1.56504e-05 7.93021e-06
  B_lm lm_title_positive 1.08269e-05 8.98566e-06
  B_lm      lm_title_net 1.06707e-05 7.40374e-06
  B_lm         kurs_lag1 7.38449e-06 2.90445e-06
  B_lm lm_title_negative 3.90995e-06 4.18422e-06
  B_lm         kurs_lag4 2.70642e-06 1.27033e-06
  B_lm         kurs_lag5 2.28782e-06 1.35029e-06
  B_

## 10. Ekspor

In [11]:
results["params"] = ""
results["cv_RMSE"] = np.nan
all_metrics = pd.concat([results, tuning], ignore_index=True)

all_metrics.to_csv(OUT_DIR / "metrics_model_B.csv", index=False)
pred_df = pd.concat(preds, ignore_index=True)
pred_df[pred_df["Split"] == "Val"].to_csv(OUT_DIR / "predictions_val_model_B.csv", index=False)
pred_df[pred_df["Split"] == "Test"].to_csv(OUT_DIR / "predictions_test_model_B.csv", index=False)
tuning.to_csv(OUT_DIR / "tuning_model_B.csv", index=False)
imp.to_csv(OUT_DIR / "importance_model_B.csv", index=False)

meta = {
    "model": "B",
    "seed": SEED,
    "sumber": ["data/export/prices_daily.parquet", "data/export/sentiment_daily.parquet"],
    "n_baris": int(len(df)),
    "split": {k: int((df["split"] == k).sum()) for k in ("train", "val", "test")},
    "fitur": {k: v for k, v in VARIANTS.items()},
    "n_model_default": len(MODELS),
    "versi": {"sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
              "lightgbm": lightgbm.__version__, "pandas": pd.__version__},
}
(OUT_DIR / "metadata_model_B.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

for f in ["metrics_model_B.csv", "predictions_val_model_B.csv", "predictions_test_model_B.csv",
          "tuning_model_B.csv", "importance_model_B.csv", "metadata_model_B.json"]:
    print(f, "|", (OUT_DIR / f).stat().st_size, "byte")

metrics_model_B.csv | 16543 byte
predictions_val_model_B.csv | 803400 byte
predictions_test_model_B.csv | 821626 byte
tuning_model_B.csv | 1275 byte
importance_model_B.csv | 3623 byte
metadata_model_B.json | 1974 byte


## 11. Ringkasan

In [12]:
summary = []
for var in ("A", "B_vader", "B_lm", "B_lex"):
    sub = all_metrics[(all_metrics["Varian"] == var) & (all_metrics["Split"] == "Test")
                      & (~all_metrics["Degenerate"])]
    best = sub.loc[sub["RMSE"].idxmin()]
    summary.append({"Varian": var, "Model": best["Model"], "n_fitur": int(best["n_fitur"]),
                    "RMSE": best["RMSE"], "MAE": best["MAE"], "DirAcc": best["DirAcc"]})
print("=== Model terbaik per varian (Test) ===")
print(pd.DataFrame(summary).to_string(index=False, float_format=lambda x: f"{x:.6g}"))

=== Model terbaik per varian (Test) ===
 Varian            Model  n_fitur       RMSE        MAE   DirAcc
      A              KNN        7 0.00298064 0.00231492 0.528409
B_vader LinearRegression       13  0.0029207  0.0022288      0.5
   B_lm    BayesianRidge       17 0.00296399 0.00226675 0.443182
  B_lex LinearRegression       23 0.00292131 0.00222007 0.556818
